In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import os

In [2]:
class CustomImageDataset(Dataset):
    def __init__(self,img_dir,transform=None):
        self.img_dir = img_dir
        self.transform = transform
        self.img_labels = []
        
        for label, class_name in enumerate(os.listdir(img_dir)):
            class_dir = os.path.join(img_dir,class_name)
            if os.path.isdir(class_dir):
                for img_name in os.listdir(class_dir):
                    self.img_labels.append((os.path.join(class_dir,img_name),label))
                    
        self.classes = set([i[1] for i in self.img_labels])
                    
    def __len__(self):
        return len(self.img_labels)
    
    def __getitem__(self, index):
        img_path, label = self.img_labels[index]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label
    
    
    
class CNN(nn.Module):
    def __init__(self, num_classes):
        super(CNN,self).__init__()
        self.cnn_kernel = 3
        self.pool_kernel = 2
        self.padding = 1
        self.stride = 2
        self.conv1 = nn.Conv2d(3,16,kernel_size=self.cnn_kernel,padding=self.padding)
        self.pool = nn.MaxPool2d(kernel_size=self.pool_kernel, stride = self.stride)
        self.conv2 = nn.Conv2d(16,32,kernel_size=self.cnn_kernel,padding=self.padding)
        self.fc1 = nn.Linear(32*64*64,128)
        self.fc2 = nn.Linear(128,num_classes)
        
    def forward(self,x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1,32*64*64)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x
    



In [3]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))])

train_dataset = CustomImageDataset(img_dir = 'C:\\Users\\adhar\\OneDrive\\Documents\\GitHub\\portfolio\\Plant Disease\\train',transform=transform)
train_loader = DataLoader(train_dataset,batch_size=512,shuffle=True)

val_dataset = CustomImageDataset(img_dir = 'C:\\Users\\adhar\\OneDrive\\Documents\\GitHub\\portfolio\\Plant Disease\\valid',transform=transform)
val_loader = DataLoader(val_dataset,batch_size=64,shuffle=False)

model = CNN(len(train_dataset.classes))


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr = 0.001)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
# device = torch.device('mps')
print(device)
model.to(device)


num_epochs = 5
loss_dct = {i:[] for i in range(num_epochs)}
print(loss_dct)

for epoch in range(num_epochs):
    model.train()
    for i, data in enumerate(train_loader,0):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs.to(device), labels.to(device))
        loss.backward()
        optimizer.step()
        loss_dct[epoch].append(loss.item())
        
        if i % 10 == 0:
            print(f'Epoch {epoch + 1}, Loss: {loss.item()}')

cuda
{0: [], 1: [], 2: [], 3: [], 4: []}
Epoch 1, Loss: 3.63736629486084
Epoch 1, Loss: 3.494194507598877
Epoch 1, Loss: 3.0299038887023926
Epoch 1, Loss: 2.221575975418091
Epoch 1, Loss: 1.7131354808807373
Epoch 1, Loss: 1.4686888456344604
Epoch 1, Loss: 1.2107782363891602
Epoch 1, Loss: 1.211172342300415
Epoch 1, Loss: 1.046863079071045
Epoch 1, Loss: 0.9612870216369629


KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), '/Users/adharshv/Documents/Career/portfolio/Plant Disease/models/modelv1.pt')

CUDA Available: True
Number of GPUs: 1
GPU Name: NVIDIA GeForce RTX 4060
